In [1]:
import torch
import yaml
from pathlib import Path
import os
import subprocess
import numpy as np
from torch_geometric.data import HeteroData
from torch_geometric.utils import subgraph

# --- Configuration for this specific experiment ---
SUBSET_RATIO = 0.2  # Use 20% of the nodes
RANDOM_SEED = 42
MODEL_TO_RUN = "RevGAT"

print("--- RevGAT Subset Experiment Setup ---")
print(f"Model: {MODEL_TO_RUN}")
print(f"Subset Ratio: {SUBSET_RATIO*100:.0f}% on the 'ogbn-arxiv' dataset")
print("-" * 40)

--- RevGAT Subset Experiment Setup ---
Model: RevGAT
Subset Ratio: 20% on the 'ogbn-arxiv' dataset
----------------------------------------


/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("config/experiments_config.yaml") as f:
    params = yaml.load(f, Loader=yaml.FullLoader)

DATASET_NAME = "ogbnarxiv"
original_data_path = Path(params["data"]["graph_dataset"][DATASET_NAME])
subset_data_path = original_data_path.parent / f"{original_data_path.stem}_subset_{int(SUBSET_RATIO*100)}pct.pt"

print(f"Original data path: {original_data_path}")
print(f"Subset will be saved to: {subset_data_path}")

Original data path: data/ogbnarxiv_heterodata.pt
Subset will be saved to: data/ogbnarxiv_heterodata_subset_20pct.pt


In [3]:
if subset_data_path.exists():
    print(f"Deleting old subset file at {subset_data_path}")
    os.remove(subset_data_path)

if not subset_data_path.exists():
    print(f"\nSubset file not found. Creating it now...")
    
    # 1. Load the original full dataset
    data = torch.load(original_data_path)
    print("Full dataset loaded successfully.")

    # Load and assign the node embeddings BEFORE creating the subset.
    print("Loading original node embeddings...")
    node_embs_type = params["node_embs"]
    path_embs = str(Path(params["data"][f"{node_embs_type}_embs"][DATASET_NAME]))
    
    if node_embs_type == "tape":
        init_x_shape = (data["paper"].num_nodes, 768)
        features = np.array(np.memmap(path_embs, mode='r', dtype=np.float16, shape=init_x_shape))
        data["paper"].x = torch.from_numpy(features).to(torch.float32)
    else:
        data["paper"].x = torch.load(path_embs).type(torch.float32)
    print(f"Original embeddings of shape {data['paper'].x.shape} loaded.")

    # 2. Select the subset of nodes
    node_type_to_subset = 'paper'
    original_num_nodes = data[node_type_to_subset].num_nodes
    num_nodes_to_keep = int(original_num_nodes * SUBSET_RATIO)
    torch.manual_seed(RANDOM_SEED)
    perm = torch.randperm(original_num_nodes)
    subset_node_indices = perm[:num_nodes_to_keep]
    
    print(f"Original '{node_type_to_subset}' nodes: {original_num_nodes}. Keeping: {num_nodes_to_keep}.")
    
    # 3. Manually build a new, clean HeteroData object
    data_subset = HeteroData()

    # 3.1. Copy the subsetted node features and attributes
    data_subset['paper'].x = data['paper'].x[subset_node_indices]
    data_subset['paper'].y = data['paper'].y[subset_node_indices]
    if 'node_year' in data['paper']:
        data_subset['paper'].node_year = data['paper'].node_year[subset_node_indices]

    # 3.2. For each edge type, get the subgraph from the sparse tensor (adj_t)
    print("Extracting subgraph from original adj_t and converting to new edge_index...")
    for edge_type in data.edge_types:
        # Check if the original data has an adj_t for this edge type
        if 'adj_t' in data[edge_type]:
            # ** FIX **
            # Convert the CSR tensor to COO format to get row and column indices
            adj_t_coo = data[edge_type].adj_t.to_sparse_coo()
            edge_index = adj_t_coo.indices()
            
            # Now, use the standard subgraph utility on the edge_index
            # This correctly re-indexes the nodes for the new smaller graph
            # We explicitly pass num_nodes to handle isolated nodes correctly.
            new_edge_index, _ = subgraph(subset_node_indices, edge_index, relabel_nodes=True, num_nodes=original_num_nodes)
            data_subset[edge_type].edge_index = new_edge_index
            # ** END FIX **

    # 4. Regenerate train/val/test splits for the new smaller graph
    print("Regenerating train/val/test splits for the subset...")
    num_subset_nodes = data_subset[node_type_to_subset].num_nodes
    split_perm = torch.randperm(num_subset_nodes)
    num_train = int(num_subset_nodes * 0.6)
    num_val = int(num_subset_nodes * 0.2)
    
    data_subset[node_type_to_subset].train_idx = split_perm[:num_train]
    data_subset[node_type_to_subset].val_idx = split_perm[num_train : num_train + num_val]
    data_subset[node_type_to_subset].test_idx = split_perm[num_train + num_val:]

    # 5. Save the new, correct subset data object
    torch.save(data_subset, subset_data_path)
    print(f"SUCCESS: Subset data saved to {subset_data_path}")
    print("\nNew Data Object Details:")
    print(data_subset)
else:
    print(f"\nSubset file already exists at {subset_data_path}. Skipping creation.")


Subset file not found. Creating it now...


/var/folders/dw/8xmb3tsd1j97gg1p6nfz7vk00000gn/T/ipykernel_37906/1884307941.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(original_data_path)
/opt/an

Full dataset loaded successfully.
Loading original node embeddings...


/var/folders/dw/8xmb3tsd1j97gg1p6nfz7vk00000gn/T/ipykernel_37906/1884307941.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data["paper"].x = torch.load(path_embs).type

Original embeddings of shape torch.Size([169343, 1024]) loaded.
Original 'paper' nodes: 169343. Keeping: 33868.
Extracting subgraph from original adj_t and converting to new edge_index...
Regenerating train/val/test splits for the subset...
SUCCESS: Subset data saved to data/ogbnarxiv_heterodata_subset_20pct.pt

New Data Object Details:
HeteroData(
  paper={
    x=[33868, 1024],
    y=[33868, 1],
    node_year=[33868, 1],
    train_idx=[20320],
    val_idx=[6773],
    test_idx=[6775],
  },
  (paper, references, paper)={ edge_index=[2, 85626] },
  (paper, shares_author_with, paper)={ edge_index=[2, 271849] },
  (paper, shares_fos_with, paper)={ edge_index=[2, 330800] },
  (paper, shares_venue_with, paper)={ edge_index=[2, 24082] }
)


In [4]:
print("\nCreating temporary config for this run...")

revgat_params = params
revgat_params['dataset'] = DATASET_NAME
revgat_params['data']['graph_dataset'][DATASET_NAME] = str(subset_data_path)
revgat_params['node_embs'] = 'preloaded_in_subset'
revgat_params['edge_type_selection'][DATASET_NAME] = ['references', 'author', 'venue', 'fos']
revgat_params['model'] = {
    'name': MODEL_TO_RUN,
    'hidden_channels': 128,
    'num_layers': 2,
    'dropout': 0.5,
    'heads': 4
}

temp_config_path = "config/temp_revgat_config.yml"
with open(temp_config_path, 'w') as f:
    yaml.dump(revgat_params, f, sort_keys=False)

print(f"Temporary config saved to: {temp_config_path}")
print("--- Config Contents ---")
with open(temp_config_path, 'r') as f:
    print(f.read())
print("-----------------------")


Creating temporary config for this run...
Temporary config saved to: config/temp_revgat_config.yml
--- Config Contents ---
seed: 1911
dataset: ogbnarxiv
edge_type_selection:
  ogbnarxiv:
  - references
  - author
  - venue
  - fos
  pubmed:
  - references
  - author
  - journal
  - mesh
node_embs: preloaded_in_subset
data:
  graph_dataset:
    ogbnarxiv: data/ogbnarxiv_heterodata_subset_20pct.pt
    pubmed: data/pubmed_heterodata.pt
  simtg_embs:
    ogbnarxiv: data/embeddings/ogbnarxiv_simtg_x_embs.pt
    pubmed: data/embeddings/pubmed_simtg_x_embs.pt
  tape_embs:
    ogbnarxiv: data/embeddings/ogbnarxiv-tape-seed0.emb
    pubmed: data/embeddings/pubmed-tape-seed0.emb
optimizer:
  lr: 0.001
  weight_decay: 0
model:
  name: RevGAT
  hidden_channels: 128
  num_layers: 2
  dropout: 0.5
  heads: 4
early_stop_threshold: 20
epochs: 500
runs: 3

-----------------------


In [5]:
print("\n" + "="*50)
print(f"STARTING EXPERIMENT: Running {MODEL_TO_RUN} on {SUBSET_RATIO*100:.0f}% of the data.")
print("The script will now take over...")
print("="*50 + "\n")

%run scripts/experiments.py --config {temp_config_path}

print("\n" + "="*50)
print("EXPERIMENT FINISHED")
print("="*50)


STARTING EXPERIMENT: Running RevGAT on 20% of the data.
The script will now take over...

Will run on: cpu.
Loading configuration from: config/temp_revgat_config.yml
Converting edge_index to sparse tensors (adj_t)...


/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:117: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path_to_dat

Loaded pre-trained node embeddings of type=preloaded_in_subset and shape=torch.Size([33868, 1024]).
Instantiating RevGAT with 4 heads.
No. parameters:  2186720


Run 00:  19%|██████▍                           | 94/500 [08:24<36:17,  5.36s/it]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:228: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for

Early stopped training for run 00 at epoch 94.


...

Early stopped training for run 01 at epoch 129.


/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/fx.py:132: UserWarning: Found function 'dropout' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  warnings.warn(f"Found function '{node.name}' with keyword "


Run 01: Best Epoch 108, Best Val Acc 0.7413, Test Acc 0.7693.
Instantiating RevGAT with 4 heads.


Run 02:  13%|████▎                             | 64/500 [05:42<38:49,  5.34s/it]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:228: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for

Early stopped training for run 02 at epoch 64.
Run 02: Best Epoch 43, Best Val Acc 0.7332, Test Acc 0.7528.
* ============================= ALL RUNS =============================
Best Val Acc: 74.13, Best Test Acc: 76.93.
Avg. Val Acc: 73.72 ± 0.41 Avg. Test Acc: 76.17 ± 0.84.

EXPERIMENT FINISHED


In [10]:
#
# Cell 4: Create a Temporary Config for the SAGE Experiment
#
print("\nCreating temporary config for this run...")

# This variable is now a generic name for the experiment parameters
exp_params = params
exp_params['dataset'] = DATASET_NAME
exp_params['data']['graph_dataset'][DATASET_NAME] = str(subset_data_path)
exp_params['node_embs'] = 'preloaded_in_subset'
exp_params['edge_type_selection'][DATASET_NAME] = ['references', 'author', 'venue', 'fos']

# Set the model to SAGE with recommended parameters
exp_params['model'] = {
    'name': 'SAGE',
    'hidden_channels': 128,
    'num_layers': 2,
    'dropout': 0.5
}

# Make the temporary config filename dynamic based on the model name
temp_config_path = "config/temp_SAGE_config.yml"
with open(temp_config_path, 'w') as f:
    yaml.dump(exp_params, f, sort_keys=False)

print(f"Temporary config saved to: {temp_config_path}")
print("--- Config Contents ---")
with open(temp_config_path, 'r') as f:
    print(f.read())
print("-----------------------")


#
# Cell 5: Run the Experiment!
#
print("\n" + "="*50)
print(f"STARTING EXPERIMENT: Running SAGE on {SUBSET_RATIO*100:.0f}% of the data.")
print("The script will now take over...")
print("="*50 + "\n")

%run scripts/experiments.py --config {temp_config_path}

print("\n" + "="*50)
print("EXPERIMENT FINISHED")
print("="*50)


Creating temporary config for this run...
Temporary config saved to: config/temp_SAGE_config.yml
--- Config Contents ---
seed: 1911
dataset: ogbnarxiv
edge_type_selection:
  ogbnarxiv:
  - references
  - author
  - venue
  - fos
  pubmed:
  - references
  - author
  - journal
  - mesh
node_embs: preloaded_in_subset
data:
  graph_dataset:
    ogbnarxiv: data/ogbnarxiv_heterodata_subset_20pct.pt
    pubmed: data/pubmed_heterodata.pt
  simtg_embs:
    ogbnarxiv: data/embeddings/ogbnarxiv_simtg_x_embs.pt
    pubmed: data/embeddings/pubmed_simtg_x_embs.pt
  tape_embs:
    ogbnarxiv: data/embeddings/ogbnarxiv-tape-seed0.emb
    pubmed: data/embeddings/pubmed-tape-seed0.emb
optimizer:
  lr: 0.001
  weight_decay: 0
model:
  name: SAGE
  hidden_channels: 128
  num_layers: 2
  dropout: 0.5
early_stop_threshold: 20
epochs: 500
runs: 3

-----------------------

STARTING EXPERIMENT: Running SAGE on 20% of the data.
The script will now take over...

Will run on: cpu.
Loading configuration from: con

/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:117: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path_to_dat

Loaded pre-trained node embeddings of type=preloaded_in_subset and shape=torch.Size([33868, 1024]).
No. parameters:  1090464


Run 00:  33%|██████████▉                      | 165/500 [02:50<05:46,  1.03s/it]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:228: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for

Early stopped training for run 00 at epoch 165.


...

Early stopped training for run 01 at epoch 141.


/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/fx.py:132: UserWarning: Found function 'dropout' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  warnings.warn(f"Found function '{node.name}' with keyword "


Run 01: Best Epoch 120, Best Val Acc 0.7630, Test Acc 0.7702.


Run 02:  26%|████████▌                        | 130/500 [02:17<06:30,  1.06s/it]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:228: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for

Early stopped training for run 02 at epoch 130.
Run 02: Best Epoch 109, Best Val Acc 0.7614, Test Acc 0.7644.
* ============================= ALL RUNS =============================
Best Val Acc: 76.30, Best Test Acc: 77.25.
Avg. Val Acc: 76.21 ± 0.08 Avg. Test Acc: 76.91 ± 0.42.

EXPERIMENT FINISHED


In [14]:
print("\nCreating temporary config for this run...")

# This variable is now a generic name for the experiment parameters
exp_params = params
exp_params['dataset'] = DATASET_NAME
exp_params['data']['graph_dataset'][DATASET_NAME] = str(subset_data_path)
exp_params['node_embs'] = 'preloaded_in_subset'
exp_params['edge_type_selection'][DATASET_NAME] = ['references', 'author', 'venue', 'fos']

# --- RECOMMENDED NEXT EXPERIMENT ---
# Set the model to RevGAT with more attention heads
exp_params['model'] = {
    'name': 'RevGAT',
    'hidden_channels': 128,
    'num_layers': 2,      # <-- Keep layers at 2
    'dropout': 0.5,
    'heads': 8          # <-- Increase heads from 4 to 8
}

# --- CHANGES FOR EXPERIMENT 3 ---
# Adjust the optimizer's learning rate
exp_params['optimizer']['lr'] = 0.0005 # <-- Lowered from 0.001

temp_config_path = f"config/temp_{MODEL_TO_RUN.lower()}_config.yml"
with open(temp_config_path, 'w') as f:
    yaml.dump(exp_params, f, sort_keys=False)

print(f"Temporary config saved to: {temp_config_path}")
print("--- Config Contents ---")
with open(temp_config_path, 'r') as f:
    print(f.read())
print("-----------------------")

print("\n" + "="*50)
print(f"STARTING EXPERIMENT: Running {MODEL_TO_RUN} on {SUBSET_RATIO*100:.0f}% of the data.")
print("The script will now take over...")
print("="*50 + "\n")

%run scripts/experiments.py --config {temp_config_path}

print("\n" + "="*50)
print("EXPERIMENT FINISHED")
print("="*50)


Creating temporary config for this run...
Temporary config saved to: config/temp_revgat_config.yml
--- Config Contents ---
seed: 1911
dataset: ogbnarxiv
edge_type_selection:
  ogbnarxiv:
  - references
  - author
  - venue
  - fos
  pubmed:
  - references
  - author
  - journal
  - mesh
node_embs: preloaded_in_subset
data:
  graph_dataset:
    ogbnarxiv: data/ogbnarxiv_heterodata_subset_20pct.pt
    pubmed: data/pubmed_heterodata.pt
  simtg_embs:
    ogbnarxiv: data/embeddings/ogbnarxiv_simtg_x_embs.pt
    pubmed: data/embeddings/pubmed_simtg_x_embs.pt
  tape_embs:
    ogbnarxiv: data/embeddings/ogbnarxiv-tape-seed0.emb
    pubmed: data/embeddings/pubmed-tape-seed0.emb
optimizer:
  lr: 0.0005
  weight_decay: 0
model:
  name: RevGAT
  hidden_channels: 128
  num_layers: 2
  dropout: 0.5
  heads: 8
early_stop_threshold: 20
epochs: 500
runs: 3

-----------------------

STARTING EXPERIMENT: Running RevGAT on 20% of the data.
The script will now take over...

Will run on: cpu.
Loading confi

/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:117: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path_to_dat

Loaded pre-trained node embeddings of type=preloaded_in_subset and shape=torch.Size([33868, 1024]).
Instantiating RevGAT with 8 heads.
No. parameters:  4372960


Run 00:  22%|██████▊                        | 109/500 [36:06<2:09:33, 19.88s/it]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:228: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for

Early stopped training for run 00 at epoch 109.
Run 00: Best Epoch 88, Best Val Acc 0.7382, Test Acc 0.7652.
Instantiating RevGAT with 8 heads.


...

Early stopped training for run 01 at epoch 90.


/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/fx.py:132: UserWarning: Found function 'dropout' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  warnings.warn(f"Found function '{node.name}' with keyword "


Run 01: Best Epoch 69, Best Val Acc 0.7362, Test Acc 0.7635.
Instantiating RevGAT with 8 heads.


Run 02:  32%|█████████▉                     | 161/500 [54:19<1:54:24, 20.25s/it]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:228: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for

Early stopped training for run 02 at epoch 161.
Run 02: Best Epoch 140, Best Val Acc 0.7400, Test Acc 0.7709.
* ============================= ALL RUNS =============================
Best Val Acc: 74.00, Best Test Acc: 77.09.
Avg. Val Acc: 73.81 ± 0.19 Avg. Test Acc: 76.65 ± 0.39.

EXPERIMENT FINISHED


In [15]:
print("\nCreating temporary config for this run...")

# This variable is now a generic name for the experiment parameters
exp_params = params
exp_params['dataset'] = DATASET_NAME
exp_params['data']['graph_dataset'][DATASET_NAME] = str(subset_data_path)
exp_params['node_embs'] = 'preloaded_in_subset'
exp_params['edge_type_selection'][DATASET_NAME] = ['references', 'author', 'venue', 'fos']

# --- NEW EXPERIMENT: Deeper Model ---
# Set the model to RevGAT with 4 layers and 4 heads
exp_params['model'] = {
    'name': 'RevGAT',
    'hidden_channels': 128,
    'num_layers': 4,      # <-- Increased layers to 4
    'dropout': 0.5,
    'heads': 4          # <-- Reverted heads to 4
}

# Make the temporary config filename dynamic based on the model name
temp_config_path = f"config/temp_{MODEL_TO_RUN.lower()}_config.yml"
with open(temp_config_path, 'w') as f:
    yaml.dump(exp_params, f, sort_keys=False)

print(f"Temporary config saved to: {temp_config_path}")
print("--- Config Contents ---")
with open(temp_config_path, 'r') as f:
    print(f.read())
print("-----------------------")


#
# Cell 5: Run the Experiment!
#
print("\n" + "="*50)
print(f"STARTING EXPERIMENT: Running {MODEL_TO_RUN} on {SUBSET_RATIO*100:.0f}% of the data.")
print("The script will now take over...")
print("="*50 + "\n")

%run scripts/experiments.py --config {temp_config_path}

print("\n" + "="*50)
print("EXPERIMENT FINISHED")
print("="*50)


Creating temporary config for this run...
Temporary config saved to: config/temp_revgat_config.yml
--- Config Contents ---
seed: 1911
dataset: ogbnarxiv
edge_type_selection:
  ogbnarxiv:
  - references
  - author
  - venue
  - fos
  pubmed:
  - references
  - author
  - journal
  - mesh
node_embs: preloaded_in_subset
data:
  graph_dataset:
    ogbnarxiv: data/ogbnarxiv_heterodata_subset_20pct.pt
    pubmed: data/pubmed_heterodata.pt
  simtg_embs:
    ogbnarxiv: data/embeddings/ogbnarxiv_simtg_x_embs.pt
    pubmed: data/embeddings/pubmed_simtg_x_embs.pt
  tape_embs:
    ogbnarxiv: data/embeddings/ogbnarxiv-tape-seed0.emb
    pubmed: data/embeddings/pubmed-tape-seed0.emb
optimizer:
  lr: 0.0005
  weight_decay: 0
model:
  name: RevGAT
  hidden_channels: 128
  num_layers: 4
  dropout: 0.5
  heads: 4
early_stop_threshold: 20
epochs: 500
runs: 3

-----------------------

STARTING EXPERIMENT: Running RevGAT on 20% of the data.
The script will now take over...

Will run on: cpu.
Loading confi

/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:117: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path_to_dat

Loaded pre-trained node embeddings of type=preloaded_in_subset and shape=torch.Size([33868, 1024]).
Instantiating RevGAT with 4 heads.


/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/fx.py:132: UserWarning: Found function 'dropout' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  warnings.warn(f"Found function '{node.name}' with keyword "
/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/fx.py:132: UserWarning: Found function 'dropout_1' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  warnings.warn(f"Found function '{node.name}' with keyword "
/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/fx.py:132: UserWarning: Found function 'dropout_2' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this 

No. parameters:  4298208


Run 00:  38%|███████████▏                 | 192/500 [1:07:18<1:47:58, 21.03s/it]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:228: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for

Early stopped training for run 00 at epoch 192.
Run 00: Best Epoch 171, Best Val Acc 0.7153, Test Acc 0.7517.
Instantiating RevGAT with 4 heads.


/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/fx.py:132: UserWarning: Found function 'dropout' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  warnings.warn(f"Found function '{node.name}' with keyword "
/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/fx.py:132: UserWarning: Found function 'dropout_1' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  warnings.warn(f"Found function '{node.name}' with keyword "
/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/fx.py:132: UserWarning: Found function 'dropout_2' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this 

KeyboardInterrupt: 


EXPERIMENT FINISHED


In [4]:
#
# Cell for GAT Experiment (with 128 hidden channels)
#

# --- 1. Define the model we want to run in this experiment ---
MODEL_TO_RUN = 'GAT'

print("\nCreating temporary config for this run...")

# --- 2. Create the experiment parameters for this specific run ---
exp_params = params
exp_params['dataset'] = DATASET_NAME
exp_params['data']['graph_dataset'][DATASET_NAME] = str(subset_data_path)
exp_params['node_embs'] = 'preloaded_in_subset'
exp_params['edge_type_selection'][DATASET_NAME] = ['references', 'author', 'venue', 'fos']

# --- NEW EXPERIMENT: Tuned GAT Model with 128 Channels ---
# This configuration keeps the model size manageable while improving other parameters.
exp_params['optimizer'] = {
    'lr': 0.001,
    'weight_decay': 0.0005  # Keep L2 regularization, it's very helpful
}
exp_params['model'] = {
    'name': MODEL_TO_RUN,
    'hidden_channels': 128, # <-- Set to 128 as you requested
    'num_layers': 2,
    'dropout': 0.5,         # Standard dropout for this model size
    'n_heads': 8            # More heads can still help find better patterns
}

# --- 3. Save the temporary config to a file ---
temp_config_path = f"config/temp_{MODEL_TO_RUN.lower()}_config.yml"
with open(temp_config_path, 'w') as f:
    yaml.dump(exp_params, f, sort_keys=False)

# --- 4. Print the config file contents for verification ---
print(f"Temporary config saved to: {temp_config_path}")
print("--- Config Contents ---")
with open(temp_config_path, 'r') as f:
    print(f.read())
print("-----------------------")


#
# Cell 5: Run the Experiment!
#
print("\n" + "="*50)
print(f"STARTING EXPERIMENT: Running {MODEL_TO_RUN} on {SUBSET_RATIO*100:.0f}% of the data.")
print("The script will now take over...")
print("="*50 + "\n")

# Use the magic command to run the script with our temporary config
%run scripts/experiments.py --config {temp_config_path}

print("\n" + "="*50)
print("EXPERIMENT FINISHED")
print("="*50)


Creating temporary config for this run...
Temporary config saved to: config/temp_gat_config.yml
--- Config Contents ---
seed: 1911
dataset: ogbnarxiv
edge_type_selection:
  ogbnarxiv:
  - references
  - author
  - venue
  - fos
  pubmed:
  - references
  - author
  - journal
  - mesh
node_embs: preloaded_in_subset
data:
  graph_dataset:
    ogbnarxiv: data/ogbnarxiv_heterodata_subset_20pct.pt
    pubmed: data/pubmed_heterodata.pt
  simtg_embs:
    ogbnarxiv: data/embeddings/ogbnarxiv_simtg_x_embs.pt
    pubmed: data/embeddings/pubmed_simtg_x_embs.pt
  tape_embs:
    ogbnarxiv: data/embeddings/ogbnarxiv-tape-seed0.emb
    pubmed: data/embeddings/pubmed-tape-seed0.emb
optimizer:
  lr: 0.001
  weight_decay: 0.0005
model:
  name: GAT
  hidden_channels: 128
  num_layers: 2
  dropout: 0.5
  n_heads: 8
early_stop_threshold: 20
epochs: 500
runs: 3

-----------------------

STARTING EXPERIMENT: Running GAT on 20% of the data.
The script will now take over...

Will run on: cpu.
Loading configur

/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:131: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path_to_dat

Loaded pre-trained node embeddings of type=preloaded_in_subset and shape=torch.Size([33868, 1024]).
Instantiating GAT with 8 heads.
No. parameters:  2956288


Run 00:   6%|█▉                              | 30/500 [04:29<1:10:14,  8.97s/it]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:242: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for

Early stopped training for run 00 at epoch 30.


...

Early stopped training for run 01 at epoch 27.


/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/fx.py:132: UserWarning: Found function 'dropout' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  warnings.warn(f"Found function '{node.name}' with keyword "
/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/fx.py:132: UserWarning: Found function 'dropout_1' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  warnings.warn(f"Found function '{node.name}' with keyword "


Run 01: Best Epoch 06, Best Val Acc 0.3321, Test Acc 0.3051.
Instantiating GAT with 8 heads.


Run 02:  62%|████████████████████▎            | 308/500 [44:26<27:42,  8.66s/it]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:242: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for

Early stopped training for run 02 at epoch 308.
Run 02: Best Epoch 287, Best Val Acc 0.6897, Test Acc 0.7080.
* ============================= ALL RUNS =============================
Best Val Acc: 68.97, Best Test Acc: 70.80.
Avg. Val Acc: 46.15 ± 19.82 Avg. Test Acc: 46.09 ± 21.65.

EXPERIMENT FINISHED


In [4]:
#
# Cell for a FAST and POWERFUL GIN Experiment (CPU-Optimized)
#

# --- 1. Define the model we want to run in this experiment ---
MODEL_TO_RUN = 'GIN'

print(f"\nCreating temporary config for a {MODEL_TO_RUN} run...")

# --- 2. Create the experiment parameters for this specific run ---
exp_params = params
exp_params['dataset'] = DATASET_NAME
exp_params['data']['graph_dataset'][DATASET_NAME] = str(subset_data_path)
exp_params['node_embs'] = 'preloaded_in_subset'
exp_params['edge_type_selection'][DATASET_NAME] = ['references', 'author', 'venue', 'fos']

# --- Standard Hyperparameters for GIN ---
# This is a robust baseline configuration that requires no tuning.
exp_params['optimizer'] = {
    'lr': 0.001,
    'weight_decay': 0.0005
}
exp_params['model'] = {
    'name': MODEL_TO_RUN,
    'hidden_channels': 128,
    'num_layers': 3,       # GIN often benefits from being slightly deeper than SAGE/GCN
    'dropout': 0.5
}
exp_params['early_stop_threshold'] = 30 # Standard patience

# --- 3. Save the temporary config to a file ---
temp_config_path = f"config/temp_{MODEL_TO_RUN.lower()}_config.yml"
with open(temp_config_path, 'w') as f:
    yaml.dump(exp_params, f, sort_keys=False)

# --- 4. Print the config for verification ---
print(f"Temporary config saved to: {temp_config_path}")
print("--- Config Contents ---")
with open(temp_config_path, 'r') as f:
    print(f.read())
print("-----------------------")


#
# Run the Experiment
#
print(f"\nSTARTING EXPERIMENT: Running {MODEL_TO_RUN}...")
%run scripts/experiments.py --config {temp_config_path}
print("\nEXPERIMENT FINISHED")


Creating temporary config for a GIN run...
Temporary config saved to: config/temp_gin_config.yml
--- Config Contents ---
seed: 1911
dataset: ogbnarxiv
edge_type_selection:
  ogbnarxiv:
  - references
  - author
  - venue
  - fos
  pubmed:
  - references
  - author
  - journal
  - mesh
node_embs: preloaded_in_subset
data:
  graph_dataset:
    ogbnarxiv: data/ogbnarxiv_heterodata_subset_20pct.pt
    pubmed: data/pubmed_heterodata.pt
  simtg_embs:
    ogbnarxiv: data/embeddings/ogbnarxiv_simtg_x_embs.pt
    pubmed: data/embeddings/pubmed_simtg_x_embs.pt
  tape_embs:
    ogbnarxiv: data/embeddings/ogbnarxiv-tape-seed0.emb
    pubmed: data/embeddings/pubmed-tape-seed0.emb
optimizer:
  lr: 0.001
  weight_decay: 0.0005
model:
  name: GIN
  hidden_channels: 128
  num_layers: 3
  dropout: 0.5
early_stop_threshold: 30
epochs: 500
runs: 3

-----------------------

STARTING EXPERIMENT: Running GIN...
Will run on: cpu.
Loading configuration from: config/temp_gin_config.yml
Converting edge_index to

/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:133: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path_to_dat

No. parameters:  189096


Run 00:  81%|██████████████████████████▌      | 403/500 [07:11<01:43,  1.07s/it]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:244: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for

Early stopped training for run 00 at epoch 403.


...

Early stopped training for run 01 at epoch 306.


/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/fx.py:132: UserWarning: Found function 'dropout' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  warnings.warn(f"Found function '{node.name}' with keyword "
/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/fx.py:132: UserWarning: Found function 'dropout_1' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  warnings.warn(f"Found function '{node.name}' with keyword "


Run 01: Best Epoch 275, Best Val Acc 0.6650, Test Acc 0.6621.
Instantiating GIN.


Run 02:  78%|█████████████████████████▋       | 390/500 [06:55<01:57,  1.07s/it]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:244: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for

Early stopped training for run 02 at epoch 390.
Run 02: Best Epoch 359, Best Val Acc 0.6743, Test Acc 0.6717.
* ============================= ALL RUNS =============================
Best Val Acc: 68.42, Best Test Acc: 67.17.
Avg. Val Acc: 67.45 ± 0.96 Avg. Test Acc: 66.79 ± 0.51.

EXPERIMENT FINISHED


In [5]:
#
# Cell 1: Train the Teacher Model (SGC) and Generate Predictions (Corrected)
#

# --- 1. Define the Teacher Model ---
TEACHER_MODEL = 'SGC'
print(f"--- Phase 1: Training the {TEACHER_MODEL} Teacher Model ---")

# --- 2. Create the config for the Teacher ---
exp_params = params
exp_params['dataset'] = DATASET_NAME
exp_params['data']['graph_dataset'][DATASET_NAME] = str(subset_data_path)
exp_params['node_embs'] = 'preloaded_in_subset'
exp_params['edge_type_selection'][DATASET_NAME] = ['references', 'author', 'venue', 'fos']
exp_params['optimizer'] = {'lr': 0.1, 'weight_decay': 0.0005}
exp_params['model'] = {
    'name': TEACHER_MODEL,
    'num_layers': 3,
    'hidden_channels': 128,
    'dropout': 0.0
}
exp_params['early_stop_threshold'] = 30
exp_params['runs'] = 1 # We only need one set of predictions

# --- 3. Save the temporary config file ---
temp_config_path = f"config/temp_{TEACHER_MODEL.lower()}_teacher_config.yml"
with open(temp_config_path, 'w') as f:
    yaml.dump(exp_params, f, sort_keys=False)

# --- 4. Run the training script ---
print(f"Running training for {TEACHER_MODEL} teacher...")
%run scripts/experiments.py --config {temp_config_path}
print("Teacher training finished.")

# --- 5. Generate and Save Pseudo-Labels ---
print("\nGenerating pseudo-labels from the trained teacher...")
# Load the data subset again
teacher_data = torch.load(str(subset_data_path))

# ===================================================================
# START: CRITICAL FIX
# Manually convert edge_index to adj_t for the reloaded data object
# ===================================================================
import torch_geometric.transforms as T
transform = T.ToSparseTensor()
print("Converting edge_index to sparse tensor (adj_t) for predictions...")
teacher_data = transform(teacher_data)
# ===================================================================
# END: CRITICAL FIX
# ===================================================================

# Get the model architecture
in_channels = teacher_data['paper'].x.shape[1]
out_channels = len(torch.unique(teacher_data['paper'].y))
teacher_model_arch = models.SGC(num_layers=3, in_channels=in_channels, out_channels=out_channels)
teacher_model_arch = to_hetero(teacher_model_arch, teacher_data.metadata(), aggr='mean')

# Load the trained weights
teacher_weights_path = f"models/{DATASET_NAME}_{TEACHER_MODEL}_run0_best.pth"
teacher_model_arch.load_state_dict(torch.load(teacher_weights_path))
teacher_model_arch.eval()

# Get predictions for ALL nodes (this will now work)
with torch.no_grad():
    pseudo_labels = teacher_model_arch(teacher_data.x_dict, teacher_data.adj_t_dict)['paper']

# Save the predictions
pseudo_labels_path = "data/embeddings/teacher_predictions.pt"
torch.save(pseudo_labels.cpu(), pseudo_labels_path)
print(f"Pseudo-labels saved to: {pseudo_labels_path}")

--- Phase 1: Training the SGC Teacher Model ---
Running training for SGC teacher...
Will run on: cpu.
Loading configuration from: config/temp_sgc_teacher_config.yml
Converting edge_index to sparse tensors (adj_t)...


/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:133: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path_to_dat

Loaded pre-trained node embeddings...
Loaded pre-trained node embeddings of type=preloaded_in_subset and shape=torch.Size([33868, 1024]).
No. parameters:  164000


Run 00:   8%|██▋                               | 39/500 [00:09<01:48,  4.25it/s]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:257: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for

Early stopped training for run 00 at epoch 39.
Run 00: Best Epoch 08, Best Val Acc 0.6535, Test Acc 0.6610.
* ============================= ALL RUNS =============================
Best Val Acc: 65.35, Best Test Acc: 66.10.
Avg. Val Acc: 65.35 ± nan Avg. Test Acc: 66.10 ± nan.
Teacher training finished.

Generating pseudo-labels from the trained teacher...
Converting edge_index to sparse tensor (adj_t) for predictions...


/var/folders/dw/8xmb3tsd1j97gg1p6nfz7vk00000gn/T/ipykernel_59634/4015344360.py:60: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  teacher_model_arch.load_state_dict(torch.loa

Pseudo-labels saved to: data/embeddings/teacher_predictions.pt


In [6]:
#
# Cell 2: Train the Student Model (RevGAT) with Augmented Features
#

# --- 1. Define the Student Model ---
STUDENT_MODEL = 'RevGAT'
print(f"\n--- Phase 2: Training the {STUDENT_MODEL} Student Model ---")

# --- 2. Create the config for the Student ---
exp_params = params
exp_params['dataset'] = DATASET_NAME
exp_params['data']['graph_dataset'][DATASET_NAME] = str(subset_data_path)
exp_params['node_embs'] = 'preloaded_in_subset'
exp_params['edge_type_selection'][DATASET_NAME] = ['references', 'author', 'venue', 'fos']

# This new key will activate the code we added to experiments.py
exp_params['augment_with_preds'] = "data/embeddings/teacher_predictions.pt"

exp_params['optimizer'] = {'lr': 0.001, 'weight_decay': 0.0005}
exp_params['model'] = {
    'name': STUDENT_MODEL,
    'hidden_channels': 128,
    'num_layers': 4, # We can now try a slightly deeper model
    'dropout': 0.5,
    'heads': 4
}
exp_params['early_stop_threshold'] = 30
exp_params['runs'] = 3

# --- 3. Save the temporary config file ---
temp_config_path = f"config/temp_{STUDENT_MODEL.lower()}_student_config.yml"
with open(temp_config_path, 'w') as f:
    yaml.dump(exp_params, f, sort_keys=False)

# --- 4. Run the training script ---
print(f"Running training for {STUDENT_MODEL} student...")
%run scripts/experiments.py --config {temp_config_path}
print("Student training finished.")


--- Phase 2: Training the RevGAT Student Model ---
Running training for RevGAT student...
Will run on: cpu.
Loading configuration from: config/temp_revgat_student_config.yml
Converting edge_index to sparse tensors (adj_t)...


/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:133: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path_to_dat

Augmenting node features with teacher predictions...
New feature shape after augmentation: torch.Size([33868, 1064])
Loaded pre-trained node embeddings...
Loaded pre-trained node embeddings of type=preloaded_in_subset and shape=torch.Size([33868, 1064]).
Instantiating RevGAT with 4 heads.


/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:165: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  teacher_preds = torch.load(pr

No. parameters:  4380128


Run 00:  44%|████████████▊                | 220/500 [1:15:21<1:35:54, 20.55s/it]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:257: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for

Early stopped training for run 00 at epoch 220.
Run 00: Best Epoch 189, Best Val Acc 0.7133, Test Acc 0.7160.
Instantiating RevGAT with 4 heads.


/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/fx.py:132: UserWarning: Found function 'dropout' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  warnings.warn(f"Found function '{node.name}' with keyword "
/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/fx.py:132: UserWarning: Found function 'dropout_1' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  warnings.warn(f"Found function '{node.name}' with keyword "
/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/fx.py:132: UserWarning: Found function 'dropout_2' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this 

KeyboardInterrupt: 

Student training finished.


In [7]:
#
# Cell 3: Train the Student Model (RevGAT) with a 2-layer, 8-head architecture
#

# --- 1. Define the Student Model ---
STUDENT_MODEL = 'RevGAT'
print(f"\n--- Phase 2 (New Config): Training the {STUDENT_MODEL} Student Model ---")

# --- 2. Create the config for the Student ---
exp_params = params
exp_params['dataset'] = DATASET_NAME
exp_params['data']['graph_dataset'][DATASET_NAME] = str(subset_data_path)
exp_params['node_embs'] = 'preloaded_in_subset'
exp_params['edge_type_selection'][DATASET_NAME] = ['references', 'author', 'venue', 'fos']

# This key activates the feature augmentation with the teacher's predictions
exp_params['augment_with_preds'] = "data/embeddings/teacher_predictions.pt"

exp_params['optimizer'] = {'lr': 0.001, 'weight_decay': 0.0005}

# --- NEW MODEL CONFIGURATION ---
exp_params['model'] = {
    'name': STUDENT_MODEL,
    'hidden_channels': 128,
    'num_layers': 2,    # <-- Set to 2 as requested
    'dropout': 0.5,
    'heads': 8          # <-- Set to 8 as requested
}
exp_params['early_stop_threshold'] = 30
exp_params['runs'] = 3

# --- 3. Save the temporary config file ---
temp_config_path = f"config/temp_{STUDENT_MODEL.lower()}_student_2L8H_config.yml"
with open(temp_config_path, 'w') as f:
    yaml.dump(exp_params, f, sort_keys=False)
    
# --- 4. Print the config for verification ---
print(f"Temporary config saved to: {temp_config_path}")
print("--- Config Contents ---")
with open(temp_config_path, 'r') as f:
    print(f.read())
print("-----------------------")

# --- 5. Run the training script ---
print(f"\nRunning training for {STUDENT_MODEL} student (2 layers, 8 heads)...")
%run scripts/experiments.py --config {temp_config_path}
print("\nStudent training finished.")


--- Phase 2 (New Config): Training the RevGAT Student Model ---
Temporary config saved to: config/temp_revgat_student_2L8H_config.yml
--- Config Contents ---
seed: 1911
dataset: ogbnarxiv
edge_type_selection:
  ogbnarxiv:
  - references
  - author
  - venue
  - fos
  pubmed:
  - references
  - author
  - journal
  - mesh
node_embs: preloaded_in_subset
data:
  graph_dataset:
    ogbnarxiv: data/ogbnarxiv_heterodata_subset_20pct.pt
    pubmed: data/pubmed_heterodata.pt
  simtg_embs:
    ogbnarxiv: data/embeddings/ogbnarxiv_simtg_x_embs.pt
    pubmed: data/embeddings/pubmed_simtg_x_embs.pt
  tape_embs:
    ogbnarxiv: data/embeddings/ogbnarxiv-tape-seed0.emb
    pubmed: data/embeddings/pubmed-tape-seed0.emb
optimizer:
  lr: 0.001
  weight_decay: 0.0005
model:
  name: RevGAT
  hidden_channels: 128
  num_layers: 2
  dropout: 0.5
  heads: 8
early_stop_threshold: 30
epochs: 500
runs: 3
augment_with_preds: data/embeddings/teacher_predictions.pt

-----------------------

Running training for Re

/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:133: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path_to_dat

Augmenting node features with teacher predictions...
New feature shape after augmentation: torch.Size([33868, 1064])
Loaded pre-trained node embeddings...
Loaded pre-trained node embeddings of type=preloaded_in_subset and shape=torch.Size([33868, 1064]).
Instantiating RevGAT with 8 heads.
No. parameters:  4536800


Run 00:  30%|█████████▎                     | 150/500 [43:12<1:40:49, 17.29s/it]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:257: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for

Early stopped training for run 00 at epoch 150.


...

Early stopped training for run 01 at epoch 134.


/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/fx.py:132: UserWarning: Found function 'dropout' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  warnings.warn(f"Found function '{node.name}' with keyword "


Run 01: Best Epoch 103, Best Val Acc 0.7304, Test Acc 0.7491.
Instantiating RevGAT with 8 heads.


Run 02:  41%|████████████▋                  | 204/500 [56:18<1:21:42, 16.56s/it]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:257: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for

Early stopped training for run 02 at epoch 204.
Run 02: Best Epoch 173, Best Val Acc 0.7347, Test Acc 0.7544.
* ============================= ALL RUNS =============================
Best Val Acc: 73.47, Best Test Acc: 75.44.
Avg. Val Acc: 73.16 ± 0.27 Avg. Test Acc: 75.05 ± 0.34.

Student training finished.


In [4]:
#
# Cell 1: Train the a more powerful Teacher Model (GraphSAGE)
#

# --- 1. Define the Teacher Model ---
TEACHER_MODEL = 'SAGE'
print(f"--- Phase 1: Training the {TEACHER_MODEL} Teacher Model ---")

# --- 2. Create the config for the Teacher ---
exp_params = params
exp_params['dataset'] = DATASET_NAME
exp_params['data']['graph_dataset'][DATASET_NAME] = str(subset_data_path)
exp_params['node_embs'] = 'preloaded_in_subset'
exp_params['edge_type_selection'][DATASET_NAME] = ['references', 'author', 'venue', 'fos']

# Use the standard, high-performing SAGE configuration
exp_params['optimizer'] = {'lr': 0.001, 'weight_decay': 0}
exp_params['model'] = {
    'name': TEACHER_MODEL,
    'hidden_channels': 128,
    'num_layers': 2,
    'dropout': 0.5
}
exp_params['early_stop_threshold'] = 30
exp_params['runs'] = 1 # We only need one run for predictions

# --- 3. Save the temporary config file ---
temp_config_path = f"config/temp_{TEACHER_MODEL.lower()}_teacher_config.yml"
with open(temp_config_path, 'w') as f:
    yaml.dump(exp_params, f, sort_keys=False)

# --- 4. Run the training script ---
print(f"Running training for {TEACHER_MODEL} teacher...")
%run scripts/experiments.py --config {temp_config_path}
print("Teacher training finished.")

# --- 5. Generate and Save Pseudo-Labels ---
print("\nGenerating pseudo-labels from the trained SAGE teacher...")
teacher_data = torch.load(str(subset_data_path))
import torch_geometric.transforms as T
transform = T.ToSparseTensor()
teacher_data = transform(teacher_data) # Ensure adj_t is present

in_channels = teacher_data['paper'].x.shape[1]
out_channels = len(torch.unique(teacher_data['paper'].y))
teacher_model_arch = models.SAGE(num_layers=2, in_channels=in_channels, out_channels=out_channels, hidden_channels=128, dropout=0.5)
teacher_model_arch = to_hetero(teacher_model_arch, teacher_data.metadata(), aggr='mean')

teacher_weights_path = f"models/{DATASET_NAME}_{TEACHER_MODEL}_run0_best.pth"
teacher_model_arch.load_state_dict(torch.load(teacher_weights_path))
teacher_model_arch.eval()

with torch.no_grad():
    pseudo_labels = teacher_model_arch(teacher_data.x_dict, teacher_data.adj_t_dict)['paper']

# Save to a new file to avoid overwriting SGC's predictions
pseudo_labels_path = "data/embeddings/sage_teacher_predictions.pt"
torch.save(pseudo_labels.cpu(), pseudo_labels_path)
print(f"SAGE pseudo-labels saved to: {pseudo_labels_path}")

--- Phase 1: Training the SAGE Teacher Model ---
Running training for SAGE teacher...
Will run on: cpu.
Loading configuration from: config/temp_sage_teacher_config.yml
Converting edge_index to sparse tensors (adj_t)...


/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:133: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path_to_dat

Loaded pre-trained node embeddings...
Loaded pre-trained node embeddings of type=preloaded_in_subset and shape=torch.Size([33868, 1024]).
No. parameters:  1090464


Run 00:   0%|                                           | 0/500 [00:00<?, ?it/s]/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_sparse/tensor.py:574: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:55.)
  return torch.sparse_csr_tensor(rowptr, col, value, self.sizes())
Run 00:  30%|█████████▊                       | 149/500 [02:44<06:28,  1.11s/it]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:257: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/b

Early stopped training for run 00 at epoch 149.
Run 00: Best Epoch 118, Best Val Acc 0.7617, Test Acc 0.7663.
* ============================= ALL RUNS =============================
Best Val Acc: 76.17, Best Test Acc: 76.63.
Avg. Val Acc: 76.17 ± nan Avg. Test Acc: 76.63 ± nan.
Teacher training finished.

Generating pseudo-labels from the trained SAGE teacher...


/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:267: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1823.)
  print(f"Avg. Val Acc: {all_runs_accs[:, 0].mean().item()*100:.2f} ± {all_runs_accs[:, 0].std().item()*100:.2f}",
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:268: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1823.)
  f"Avg. Test Acc: {all_runs_accs[:, 1].mean().item()*100:.2f} ± {all_runs_accs[:, 1].std().item()*100:.2f}.")
/var/folders/dw/8xmb3tsd1j97gg1p6nfz7vk00000gn/T/ipykernel_880

SAGE pseudo-labels saved to: data/embeddings/sage_teacher_predictions.pt


In [5]:
#
# Cell 2: Train the RevGAT Student with guidance from the SAGE Teacher
#

# --- 1. Define the Student Model ---
STUDENT_MODEL = 'RevGAT'
print(f"\n--- Phase 2: Training {STUDENT_MODEL} with a SAGE teacher ---")

# --- 2. Create the config for the Student ---
exp_params = params
exp_params['dataset'] = DATASET_NAME
exp_params['data']['graph_dataset'][DATASET_NAME] = str(subset_data_path)
exp_params['node_embs'] = 'preloaded_in_subset'
exp_params['edge_type_selection'][DATASET_NAME] = ['references', 'author', 'venue', 'fos']

# This key now points to the SAGE predictions
exp_params['augment_with_preds'] = "data/embeddings/sage_teacher_predictions.pt"

exp_params['optimizer'] = {'lr': 0.001, 'weight_decay': 0.0005}

# Use the best RevGAT architecture we found
exp_params['model'] = {
    'name': STUDENT_MODEL,
    'hidden_channels': 128,
    'num_layers': 2,
    'dropout': 0.5,
    'heads': 8
}
exp_params['early_stop_threshold'] = 30
exp_params['runs'] = 3

# --- 3. Save the temporary config file ---
temp_config_path = f"config/temp_{STUDENT_MODEL.lower()}_student_sage_teacher_config.yml"
with open(temp_config_path, 'w') as f:
    yaml.dump(exp_params, f, sort_keys=False)

# --- 4. Run the training script ---
print(f"Running training for {STUDENT_MODEL} student (SAGE teacher)...")
%run scripts/experiments.py --config {temp_config_path}
print("\nStudent training finished.")


--- Phase 2: Training RevGAT with a SAGE teacher ---
Running training for RevGAT student (SAGE teacher)...
Will run on: cpu.
Loading configuration from: config/temp_revgat_student_sage_teacher_config.yml
Converting edge_index to sparse tensors (adj_t)...


/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:133: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path_to_dat

Augmenting node features with teacher predictions...
New feature shape after augmentation: torch.Size([33868, 1064])
Loaded pre-trained node embeddings...
Loaded pre-trained node embeddings of type=preloaded_in_subset and shape=torch.Size([33868, 1064]).
Instantiating RevGAT with 8 heads.
No. parameters:  4536800


Run 00:  34%|██████████▋                    | 172/500 [54:27<1:43:51, 19.00s/it]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:257: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for

Early stopped training for run 00 at epoch 172.
Run 00: Best Epoch 141, Best Val Acc 0.7350, Test Acc 0.7597.
Instantiating RevGAT with 8 heads.


/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/fx.py:132: UserWarning: Found function 'dropout' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  warnings.warn(f"Found function '{node.name}' with keyword "
Run 01:   5%|█▋                              | 26/500 [09:20<2:50:13, 21.55s/it]


KeyboardInterrupt: 


Student training finished.


In [7]:
#
# Cell 1: Train SAGE Teacher on 40% Subset (ALL Edge Types)
#

TEACHER_MODEL = 'SAGE'
print(f"\n--- Phase 1 (40% Data, All Edges): Training the {TEACHER_MODEL} Teacher ---")

exp_params = params
exp_params['dataset'] = DATASET_NAME
exp_params['data']['graph_dataset'][DATASET_NAME] = str(subset_data_path) # Use 40% subset
exp_params['node_embs'] = 'preloaded_in_subset'

# --- USE ALL FOUR EDGE TYPES ---
exp_params['edge_type_selection'][DATASET_NAME] = ['references', 'author', 'venue', 'fos']

# Clean up student keys
exp_params['augment_with_preds'] = False
if 'distillation_teacher_preds' in exp_params: del exp_params['distillation_teacher_preds']
if 'distillation_alpha' in exp_params.get('model', {}): del exp_params['model']['distillation_alpha']

# SAGE config
exp_params['optimizer'] = {'lr': 0.001, 'weight_decay': 0}
exp_params['model'] = {'name': TEACHER_MODEL, 'hidden_channels': 128, 'num_layers': 2, 'dropout': 0.5}
exp_params['early_stop_threshold'] = 30
exp_params['runs'] = 1

# Save config and run
temp_config_path = f"config/temp_{TEACHER_MODEL.lower()}_teacher_40pct_alledges_config.yml"
PREDS_PATH = f"data/embeddings/sage_teacher_40pct_alledges_predictions.pt" # New prediction file

with open(temp_config_path, 'w') as f: yaml.dump(exp_params, f, sort_keys=False)
%run scripts/experiments.py --config {temp_config_path} --generate_preds_path {PREDS_PATH}

print(f"\nTeacher training finished. Predictions saved to: {PREDS_PATH}")

--- Phase 1: Training the SAGE Teacher Model ---
Running training for SAGE teacher...
Will run on: cpu.
Loading configuration from: config/temp_sage_teacher_config.yml
Converting edge_index to sparse tensors (adj_t)...


/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:158: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data["paper"].x = torch.load(

Loaded pre-trained node embeddings...
Loaded pre-trained node embeddings of type=preloaded_in_subset and shape=torch.Size([33868, 1024]).
No. parameters:  1090464


Run 00:  30%|█████████▉                       | 151/500 [02:31<05:50,  1.00s/it]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:295: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for

Early stopped training for run 00 at epoch 151.
Run 00: Best Epoch 120, Best Val Acc 0.7604, Test Acc 0.7662.
* ============================= ALL RUNS =============================
Best Val Acc: 76.04, Best Test Acc: 76.62.
Avg. Val Acc: 76.04 ± nan Avg. Test Acc: 76.62 ± nan.
Teacher training finished.

Generating pseudo-labels from the trained SAGE teacher...


/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:305: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1823.)
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:306: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1823.)
/var/folders/dw/8xmb3tsd1j97gg1p6nfz7vk00000gn/T/ipykernel_88092/979165062.py:53: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will

SAGE pseudo-labels saved to: data/embeddings/sage_teacher_predictions.pt


In [8]:
#
# Cell 3: Train RevGAT Student with Knowledge Distillation Loss
#

# --- 1. Define the Student Model ---
STUDENT_MODEL = 'RevGAT'
print(f"\n--- Phase 2 (Distillation Loss): Training {STUDENT_MODEL} with a SAGE teacher ---")

# --- 2. Create the config for the Student ---
exp_params = params
exp_params['dataset'] = DATASET_NAME
exp_params['data']['graph_dataset'][DATASET_NAME] = str(subset_data_path)
exp_params['node_embs'] = 'preloaded_in_subset'
exp_params['edge_type_selection'][DATASET_NAME] = ['references', 'author', 'venue', 'fos']

# We are NOT augmenting features this time. This key should be gone or False.
exp_params['augment_with_preds'] = False 

# --- NEW KEYS FOR KNOWLEDGE DISTILLATION ---
# This key points to the teacher's predictions
exp_params['distillation_teacher_preds'] = "data/embeddings/sage_teacher_predictions.pt"

exp_params['optimizer'] = {'lr': 0.001, 'weight_decay': 0.0005}

exp_params['model'] = {
    'name': STUDENT_MODEL,
    'hidden_channels': 128,
    'num_layers': 2,
    'dropout': 0.5,
    'heads': 8,
    # This key controls the balance between the true loss and the teacher loss
    'distillation_alpha': 0.5 # 50% from true labels, 50% from teacher hints
}
exp_params['early_stop_threshold'] = 30
exp_params['runs'] = 3

# --- 3. Save the temporary config file ---
temp_config_path = f"config/temp_{STUDENT_MODEL.lower()}_student_distill_config.yml"
with open(temp_config_path, 'w') as f:
    yaml.dump(exp_params, f, sort_keys=False)

# --- 4. Run the training script ---
print(f"Running training for {STUDENT_MODEL} student (Distillation Loss)...")
%run scripts/experiments.py --config {temp_config_path}
print("\nStudent training finished.")


--- Phase 2 (Distillation Loss): Training RevGAT with a SAGE teacher ---
Running training for RevGAT student (Distillation Loss)...
Will run on: cpu.
Loading configuration from: config/temp_revgat_student_distill_config.yml
Converting edge_index to sparse tensors (adj_t)...
Loaded pre-trained node embeddings...
Loaded pre-trained node embeddings of type=preloaded_in_subset and shape=torch.Size([33868, 1024]).
Loading teacher predictions for distillation...
Instantiating RevGAT with 8 heads.


/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:158: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data["paper"].x = torch.load(

No. parameters:  4372960


Run 00:  18%|█████▊                          | 90/500 [28:43<2:10:50, 19.15s/it]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:295: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for

Early stopped training for run 00 at epoch 90.


...

Run 00: Best Epoch 59, Best Val Acc 0.7370, Test Acc 0.7634.
Instantiating RevGAT with 8 heads.


Run 01:  38%|███████████                  | 191/500 [1:03:14<1:42:18, 19.87s/it]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:295: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for

Early stopped training for run 01 at epoch 191.
Run 01: Best Epoch 160, Best Val Acc 0.7441, Test Acc 0.7734.
Instantiating RevGAT with 8 heads.


/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/fx.py:132: UserWarning: Found function 'dropout' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  warnings.warn(f"Found function '{node.name}' with keyword "
Run 02:  25%|███████▊                       | 127/500 [38:19<1:52:33, 18.11s/it]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:295: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the 

Early stopped training for run 02 at epoch 127.
Run 02: Best Epoch 96, Best Val Acc 0.7416, Test Acc 0.7694.
* ============================= ALL RUNS =============================
Best Val Acc: 74.41, Best Test Acc: 77.34.
Avg. Val Acc: 74.09 ± 0.36 Avg. Test Acc: 76.88 ± 0.51.

Student training finished.


In [9]:
#
# Cell for a standalone GraphSAGE Experiment
#

# --- 1. Define the model we want to run in this experiment ---
MODEL_TO_RUN = 'SAGE'

print(f"\nCreating temporary config for a standalone {MODEL_TO_RUN} run...")

# --- 2. Create a clean configuration for SAGE ---
exp_params = params
exp_params['dataset'] = DATASET_NAME
exp_params['data']['graph_dataset'][DATASET_NAME] = str(subset_data_path)
exp_params['node_embs'] = 'preloaded_in_subset'
exp_params['edge_type_selection'][DATASET_NAME] = ['references', 'author', 'venue', 'fos']

# --- Explicitly disable any teacher-student learning ---
# This ensures a clean, standalone run of the SAGE model.
exp_params['augment_with_preds'] = False
if 'distillation_teacher_preds' in exp_params:
    del exp_params['distillation_teacher_preds']
if 'distillation_alpha' in exp_params.get('model', {}):
    del exp_params['model']['distillation_alpha']

# --- Standard, untuned hyperparameters for SAGE ---
exp_params['optimizer'] = {
    'lr': 0.001,
    'weight_decay': 0  # SAGE often works well without weight decay initially
}
exp_params['model'] = {
    'name': MODEL_TO_RUN,
    'hidden_channels': 128,
    'num_layers': 2,
    'dropout': 0.5
}
exp_params['early_stop_threshold'] = 30
exp_params['runs'] = 3  # Set to 3 for a proper average and standard deviation

# --- 3. Save the temporary config to a file ---
temp_config_path = f"config/temp_{MODEL_TO_RUN.lower()}_standalone_config.yml"
with open(temp_config_path, 'w') as f:
    yaml.dump(exp_params, f, sort_keys=False)

# --- 4. Print the config for verification ---
print(f"Temporary config saved to: {temp_config_path}")
print("--- Config Contents ---")
with open(temp_config_path, 'r') as f:
    print(f.read())
print("-----------------------")


#
# Run the Experiment
#
print(f"\nSTARTING EXPERIMENT: Running {MODEL_TO_RUN} on {SUBSET_RATIO*100:.0f}% of the data.")
%run scripts/experiments.py --config {temp_config_path}
print("\nEXPERIMENT FINISHED")



Creating temporary config for a standalone SAGE run...
Temporary config saved to: config/temp_sage_standalone_config.yml
--- Config Contents ---
seed: 1911
dataset: ogbnarxiv
edge_type_selection:
  ogbnarxiv:
  - references
  - author
  - venue
  - fos
  pubmed:
  - references
  - author
  - journal
  - mesh
node_embs: preloaded_in_subset
data:
  graph_dataset:
    ogbnarxiv: data/ogbnarxiv_heterodata_subset_20pct.pt
    pubmed: data/pubmed_heterodata.pt
  simtg_embs:
    ogbnarxiv: data/embeddings/ogbnarxiv_simtg_x_embs.pt
    pubmed: data/embeddings/pubmed_simtg_x_embs.pt
  tape_embs:
    ogbnarxiv: data/embeddings/ogbnarxiv-tape-seed0.emb
    pubmed: data/embeddings/pubmed-tape-seed0.emb
optimizer:
  lr: 0.001
  weight_decay: 0
model:
  name: SAGE
  hidden_channels: 128
  num_layers: 2
  dropout: 0.5
early_stop_threshold: 30
epochs: 500
runs: 3
augment_with_preds: false

-----------------------

STARTING EXPERIMENT: Running SAGE on 20% of the data.
Will run on: cpu.
Loading configu

/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:158: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data["paper"].x = torch.load(

Loaded pre-trained node embeddings...
Loaded pre-trained node embeddings of type=preloaded_in_subset and shape=torch.Size([33868, 1024]).
No. parameters:  1090464


Run 00:  32%|██████████▌                      | 160/500 [01:40<03:32,  1.60it/s]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:295: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for

Early stopped training for run 00 at epoch 160.


...

Early stopped training for run 01 at epoch 150.


/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/fx.py:132: UserWarning: Found function 'dropout' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  warnings.warn(f"Found function '{node.name}' with keyword "


Run 01: Best Epoch 119, Best Val Acc 0.7610, Test Acc 0.7658.


Run 02:  36%|████████████                     | 182/500 [01:58<03:26,  1.54it/s]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:295: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for

Early stopped training for run 02 at epoch 182.
Run 02: Best Epoch 151, Best Val Acc 0.7616, Test Acc 0.7640.
* ============================= ALL RUNS =============================
Best Val Acc: 76.16, Best Test Acc: 76.78.
Avg. Val Acc: 76.11 ± 0.04 Avg. Test Acc: 76.59 ± 0.19.

EXPERIMENT FINISHED


In [4]:
import torch
import yaml
from pathlib import Path
import os
import subprocess
import numpy as np
from torch_geometric.data import HeteroData
from torch_geometric.utils import subgraph

# --- Configuration for this specific experiment ---
SUBSET_RATIO = 0.4  # Use 20% of the nodes
RANDOM_SEED = 42
MODEL_TO_RUN = "RevGAT"

print("--- RevGAT Subset Experiment Setup ---")
print(f"Model: {MODEL_TO_RUN}")
print(f"Subset Ratio: {SUBSET_RATIO*100:.0f}% on the 'ogbn-arxiv' dataset")
print("-" * 40)

--- RevGAT Subset Experiment Setup ---
Model: RevGAT
Subset Ratio: 40% on the 'ogbn-arxiv' dataset
----------------------------------------


In [5]:
with open("config/experiments_config.yaml") as f:
    params = yaml.load(f, Loader=yaml.FullLoader)

DATASET_NAME = "ogbnarxiv"
original_data_path = Path(params["data"]["graph_dataset"][DATASET_NAME])
subset_data_path = original_data_path.parent / f"{original_data_path.stem}_subset_{int(SUBSET_RATIO*100)}pct.pt"

print(f"Original data path: {original_data_path}")
print(f"Subset will be saved to: {subset_data_path}")

Original data path: data/ogbnarxiv_heterodata.pt
Subset will be saved to: data/ogbnarxiv_heterodata_subset_40pct.pt


In [6]:
if subset_data_path.exists():
    print(f"Deleting old subset file at {subset_data_path}")
    os.remove(subset_data_path)

if not subset_data_path.exists():
    print(f"\nSubset file not found. Creating it now...")
    
    # 1. Load the original full dataset
    data = torch.load(original_data_path)
    print("Full dataset loaded successfully.")

    # Load and assign the node embeddings BEFORE creating the subset.
    print("Loading original node embeddings...")
    node_embs_type = params["node_embs"]
    path_embs = str(Path(params["data"][f"{node_embs_type}_embs"][DATASET_NAME]))
    
    if node_embs_type == "tape":
        init_x_shape = (data["paper"].num_nodes, 768)
        features = np.array(np.memmap(path_embs, mode='r', dtype=np.float16, shape=init_x_shape))
        data["paper"].x = torch.from_numpy(features).to(torch.float32)
    else:
        data["paper"].x = torch.load(path_embs).type(torch.float32)
    print(f"Original embeddings of shape {data['paper'].x.shape} loaded.")

    # 2. Select the subset of nodes
    node_type_to_subset = 'paper'
    original_num_nodes = data[node_type_to_subset].num_nodes
    num_nodes_to_keep = int(original_num_nodes * SUBSET_RATIO)
    torch.manual_seed(RANDOM_SEED)
    perm = torch.randperm(original_num_nodes)
    subset_node_indices = perm[:num_nodes_to_keep]
    
    print(f"Original '{node_type_to_subset}' nodes: {original_num_nodes}. Keeping: {num_nodes_to_keep}.")
    
    # 3. Manually build a new, clean HeteroData object
    data_subset = HeteroData()

    # 3.1. Copy the subsetted node features and attributes
    data_subset['paper'].x = data['paper'].x[subset_node_indices]
    data_subset['paper'].y = data['paper'].y[subset_node_indices]
    if 'node_year' in data['paper']:
        data_subset['paper'].node_year = data['paper'].node_year[subset_node_indices]

    # 3.2. For each edge type, get the subgraph from the sparse tensor (adj_t)
    print("Extracting subgraph from original adj_t and converting to new edge_index...")
    for edge_type in data.edge_types:
        # Check if the original data has an adj_t for this edge type
        if 'adj_t' in data[edge_type]:
            # ** FIX **
            # Convert the CSR tensor to COO format to get row and column indices
            adj_t_coo = data[edge_type].adj_t.to_sparse_coo()
            edge_index = adj_t_coo.indices()
            
            # Now, use the standard subgraph utility on the edge_index
            # This correctly re-indexes the nodes for the new smaller graph
            # We explicitly pass num_nodes to handle isolated nodes correctly.
            new_edge_index, _ = subgraph(subset_node_indices, edge_index, relabel_nodes=True, num_nodes=original_num_nodes)
            data_subset[edge_type].edge_index = new_edge_index
            # ** END FIX **

    # 4. Regenerate train/val/test splits for the new smaller graph
    print("Regenerating train/val/test splits for the subset...")
    num_subset_nodes = data_subset[node_type_to_subset].num_nodes
    split_perm = torch.randperm(num_subset_nodes)
    num_train = int(num_subset_nodes * 0.6)
    num_val = int(num_subset_nodes * 0.2)
    
    data_subset[node_type_to_subset].train_idx = split_perm[:num_train]
    data_subset[node_type_to_subset].val_idx = split_perm[num_train : num_train + num_val]
    data_subset[node_type_to_subset].test_idx = split_perm[num_train + num_val:]

    # 5. Save the new, correct subset data object
    torch.save(data_subset, subset_data_path)
    print(f"SUCCESS: Subset data saved to {subset_data_path}")
    print("\nNew Data Object Details:")
    print(data_subset)
else:
    print(f"\nSubset file already exists at {subset_data_path}. Skipping creation.")


Subset file not found. Creating it now...


/var/folders/dw/8xmb3tsd1j97gg1p6nfz7vk00000gn/T/ipykernel_37906/1884307941.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(original_data_path)


Full dataset loaded successfully.
Loading original node embeddings...


/var/folders/dw/8xmb3tsd1j97gg1p6nfz7vk00000gn/T/ipykernel_37906/1884307941.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data["paper"].x = torch.load(path_embs).type

Original embeddings of shape torch.Size([169343, 1024]) loaded.
Original 'paper' nodes: 169343. Keeping: 67737.
Extracting subgraph from original adj_t and converting to new edge_index...
Regenerating train/val/test splits for the subset...
SUCCESS: Subset data saved to data/ogbnarxiv_heterodata_subset_40pct.pt

New Data Object Details:
HeteroData(
  paper={
    x=[67737, 1024],
    y=[67737, 1],
    node_year=[67737, 1],
    train_idx=[40642],
    val_idx=[13547],
    test_idx=[13548],
  },
  (paper, references, paper)={ edge_index=[2, 364638] },
  (paper, shares_author_with, paper)={ edge_index=[2, 1097597] },
  (paper, shares_fos_with, paper)={ edge_index=[2, 1324422] },
  (paper, shares_venue_with, paper)={ edge_index=[2, 98732] }
)


In [7]:
import torch
import yaml
from pathlib import Path
import os
import subprocess
import numpy as np
from torch_geometric.data import HeteroData
from torch_geometric.utils import subgraph

# --- Configuration for this specific experiment ---
SUBSET_RATIO = 0.2  # Use 20% of the nodes
RANDOM_SEED = 42
MODEL_TO_RUN = "RevGAT"

print("--- RevGAT Subset Experiment Setup ---")
print(f"Model: {MODEL_TO_RUN}")
print(f"Subset Ratio: {SUBSET_RATIO*100:.0f}% on the 'ogbn-arxiv' dataset")
print("-" * 40)

--- RevGAT Subset Experiment Setup ---
Model: RevGAT
Subset Ratio: 20% on the 'ogbn-arxiv' dataset
----------------------------------------


In [8]:
with open("config/experiments_config.yaml") as f:
    params = yaml.load(f, Loader=yaml.FullLoader)

DATASET_NAME = "ogbnarxiv"
# use v2 full graph instead of original
original_data_path = Path("data/ogbnarxiv_heterodata_v2.pt")

# save as new name to avoid overwrite of old subset
subset_data_path = original_data_path.parent / f"{original_data_path.stem}_subset_{int(SUBSET_RATIO*100)}pct_v2.pt"

print(f"Original data path: {original_data_path}")
print(f"Subset will be saved to: {subset_data_path}")

Original data path: data/ogbnarxiv_heterodata_v2.pt
Subset will be saved to: data/ogbnarxiv_heterodata_v2_subset_20pct_v2.pt


In [9]:
if subset_data_path.exists():
    print(f"Deleting old subset file at {subset_data_path}")
    os.remove(subset_data_path)

if not subset_data_path.exists():
    print(f"\nSubset file not found. Creating it now...")
    
    # 1. Load the original full dataset
    data = torch.load(original_data_path)
    print("Full dataset loaded successfully.")

    # Load and assign the node embeddings BEFORE creating the subset.
    # print("Loading original node embeddings...")
    # node_embs_type = params["node_embs"]
    # path_embs = str(Path(params["data"][f"{node_embs_type}_embs"][DATASET_NAME]))
    
    # if node_embs_type == "tape":
    #     init_x_shape = (data["paper"].num_nodes, 768)
    #     features = np.array(np.memmap(path_embs, mode='r', dtype=np.float16, shape=init_x_shape))
    #     data["paper"].x = torch.from_numpy(features).to(torch.float32)
    # else:
    #     data["paper"].x = torch.load(path_embs).type(torch.float32)
    # print(f"Original embeddings of shape {data['paper'].x.shape} loaded.")

    # 2. Select the subset of nodes
    node_type_to_subset = 'paper'
    original_num_nodes = data[node_type_to_subset].num_nodes
    num_nodes_to_keep = int(original_num_nodes * SUBSET_RATIO)
    torch.manual_seed(RANDOM_SEED)
    perm = torch.randperm(original_num_nodes)
    subset_node_indices = perm[:num_nodes_to_keep]
    
    print(f"Original '{node_type_to_subset}' nodes: {original_num_nodes}. Keeping: {num_nodes_to_keep}.")
    
    # 3. Manually build a new, clean HeteroData object
    data_subset = HeteroData()

    # 3.1. Copy the subsetted node features and attributes
    data_subset['paper'].x = data['paper'].x[subset_node_indices]
    data_subset['paper'].y = data['paper'].y[subset_node_indices]
    if 'node_year' in data['paper']:
        data_subset['paper'].node_year = data['paper'].node_year[subset_node_indices]

    # 3.2. For each edge type, get the subgraph from the sparse tensor (adj_t)
    print("Extracting subgraph from original adj_t and converting to new edge_index...")
    for edge_type in data.edge_types:
        if hasattr(data[edge_type], "adj_t"):
            row, col, _ = data[edge_type].adj_t.coo()
            edge_index = torch.stack([row, col], dim=0)
    
            new_edge_index, _ = subgraph(
                subset_node_indices,
                edge_index,
                relabel_nodes=True,
                num_nodes=original_num_nodes
            )
    
            data_subset[edge_type].edge_index = new_edge_index

            # ** END FIX **

    # 4. Regenerate train/val/test splits for the new smaller graph
    print("Regenerating train/val/test splits for the subset...")
    num_subset_nodes = data_subset[node_type_to_subset].num_nodes
    split_perm = torch.randperm(num_subset_nodes)
    num_train = int(num_subset_nodes * 0.6)
    num_val = int(num_subset_nodes * 0.2)
    
    data_subset[node_type_to_subset].train_idx = split_perm[:num_train]
    data_subset[node_type_to_subset].val_idx = split_perm[num_train : num_train + num_val]
    data_subset[node_type_to_subset].test_idx = split_perm[num_train + num_val:]
    
    data_subset['paper'].orig_index = subset_node_indices  # <--- HERE
    
    torch.save(data_subset, subset_data_path)

    print(f"SUCCESS: Subset data saved to {subset_data_path}")
    print("\nNew Data Object Details:")
    print(data_subset)
else:
    print(f"\nSubset file already exists at {subset_data_path}. Skipping creation.")


Subset file not found. Creating it now...


/var/folders/dw/8xmb3tsd1j97gg1p6nfz7vk00000gn/T/ipykernel_23895/944364860.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(original_data_path)


Full dataset loaded successfully.
Original 'paper' nodes: 169343. Keeping: 33868.
Extracting subgraph from original adj_t and converting to new edge_index...
Regenerating train/val/test splits for the subset...
SUCCESS: Subset data saved to data/ogbnarxiv_heterodata_v2_subset_20pct_v2.pt

New Data Object Details:
HeteroData(
  paper={
    x=[33868, 128],
    y=[33868, 1],
    node_year=[33868, 1],
    train_idx=[20320],
    val_idx=[6773],
    test_idx=[6775],
    orig_index=[33868],
  },
  (paper, references, paper)={ edge_index=[2, 85626] },
  (paper, shares_author_with, paper)={ edge_index=[2, 271849] },
  (paper, shares_fos_with, paper)={ edge_index=[2, 330800] },
  (paper, shares_venue_with, paper)={ edge_index=[2, 24082] }
)
